In [ ]:
# preparation 结果材料 → GLM 构题与审核 → 新训练表 → 逐条预览。
# 数据迁移后首次运行请先 Restart Kernel；手动执行本格会调用模型。
import os
import sys
import asyncio
from pathlib import Path
from base64 import b64encode
from html import escape
import pandas as pd
from IPython.display import HTML, display

PROJECT = Path("/yzp/zhaozy/yangzepeng/0905/demiwtg")
sys.path.insert(0, str(PROJECT))
DATA_ROOT = PROJECT.parent
os.environ["DEMIWTG_DATASETS_ROOT"] = str(DATA_ROOT)
from demiflow import data
from demiflow.lance.blobs import BlobRef
from curation.t2i.t2i_train_pipeline import config, run_pipeline

# 输入：直接读取 preparation 结果表；固定版本，按审核状态与 CONCEPTS 筛选。
# 文章只取已审核记录；图片只取审核保留且可交付的概念关系。
ARTICLE_SOURCES = [{"uri": "demiwtg/preparation/datasets/articles.lance", "version": 4}]
VISUAL_SOURCES = [{"uri": "demiwtg/preparation/datasets/images.lance", "version": 5}]
CONCEPTS = ["莜面栲栳栳", "猪鼻龟"]
MODEL = "glm/glm-5.3-flash"
RUN_ID = "glm53_training_20260925_01"  # 换来源、概念或配置时改名；相同运行可续跑
OUTPUT_TABLE_URI = 'demiwtg/curation/t2i/datasets/training_samples__glm53_training_20260925_01.lance'  # 最终结果表，可直接修改。
WRITE_MODE = 'overwrite'  # 'overwrite' 覆盖目标表；'append' 追加本次结果。
RUN_DIR = DATA_ROOT / "demiwtg/curation/t2i/datasets" / RUN_ID
CONFIG = config(
    mode="modelhub", concepts=CONCEPTS, samples_per_concept=2,
    max_target_cycles=1, max_training_attempts=4,
    reference_batch_size=4, max_reference_images=8,
    model={"model": MODEL, "max_output_tokens": 16384, "max_calls": 24},
)

# 2–3. 正式 pipeline：读 preparation 结果表 → 构题 → 校验 → GLM 审核 → 写训练表。
# 只导出通过审核的条目；表直接放 datasets/，RUN_DIR 仅用于命名。
state = await asyncio.to_thread(
    run_pipeline, RUN_DIR, ARTICLE_SOURCES, CONFIG, visual_sources=VISUAL_SOURCES, target_uri=OUTPUT_TABLE_URI, write_mode=WRITE_MODE,
)
output = state["stages"]["training_samples"]["dataset_ref"]
TABLE_URI = DATA_ROOT / output["relative_uri"]
VERSION = output["lance_version"]
print(f"训练表：{TABLE_URI}，版本：{VERSION}")

# 4. 从刚写入的固定版本读回；一行一条训练数据，点击图片放大/收起。
samples = data.read_lance(str(TABLE_URI), version=VERSION).take_all()
print(f"通过审核：{len(samples)} 条（每个概念最多 2 条）")
if not samples:
    print("没有通过审核的条目；失败记录表：", state["stages"]["incomplete"]["dataset_ref"]["relative_uri"])

preview_rows = []
for sample in samples:
    row = {"样本 ID": escape(sample["sample_id"]), "概念": escape(sample["concept"])}
    for label, content in [
        ("训练输入（指令 / 文字 / 参考图）", sample["input_content"]),
        ("监督目标", [{**sample["target"], "type": "image_blob"}]),
    ]:
        parts = []
        for part in content:
            if part["type"] == "text":
                parts.append(f'<div style="white-space:pre-wrap">{escape(part["text"])}</div>')
            else:
                raw = BlobRef(**part["blob_ref"]).read(DATA_ROOT)
                url = f'data:{escape(part["mime"])};base64,{b64encode(raw).decode()}'
                parts.append(f'<details><summary><img src="{url}" alt="点击预览图片"></summary></details>')
        row[label] = "".join(parts)
    preview_rows.append(row)

preview = pd.DataFrame(preview_rows, columns=["样本 ID", "概念", "训练输入（指令 / 文字 / 参考图）", "监督目标"])
display(HTML("""<style>
#training-preview td {vertical-align:top; text-align:left; max-width:720px; overflow-wrap:anywhere;}
#training-preview summary {cursor:pointer;}
#training-preview img {width:160px; max-width:100%;}
#training-preview details[open] img {width:640px;}
</style>""" + preview.to_html(index=False, escape=False, table_id="training-preview")))
